# GENREation: Music Genre Classification Using CNN

**Project objective:** Predict the musical genre of an audio track using audio features and a Convolutional Neural Network (CNN).

This notebook documents the complete workflow: GTZAN dataset inspection, audio preprocessing, MFCC feature extraction, stratified train/validation split, CNN construction, model training, evaluation, saving the trained model, and testing an unseen audio file.

**Technology stack:** Python, Librosa, NumPy, Pandas, Matplotlib, Seaborn, Scikit-learn, TensorFlow/Keras.

**Genres:** Blues, Classical, Country, Disco, Hip-Hop, Jazz, Metal, Pop, Reggae, Rock.

> Run the notebook from the project root directory. Training requires the GTZAN dataset and may take some time. The notebook does not assume a particular accuracy; all reported metrics are generated by the executed evaluation cells.

## 1. Import libraries and configure paths

The notebook uses the same shared MFCC extraction function as the Streamlit application (`src/audio_features.py`). Keeping preprocessing consistent is important: the training and inference pipelines must produce identically shaped features.

In [ ]:
from pathlib import Path
import json
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import tensorflow as tf

from IPython.display import Audio, display
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

from src.audio_features import (
    extract_mfcc,
    SAMPLE_RATE,
    DURATION_SECONDS,
    N_MFCC,
    MAX_FRAMES,
)

warnings.filterwarnings("ignore")
SEED = 42
GENRES = [
    "blues", "classical", "country", "disco", "hiphop",
    "jazz", "metal", "pop", "reggae", "rock"
]
DATA_DIR = Path("Data")  # Change this if your dataset is stored elsewhere.
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Librosa:", librosa.__version__)
print("Dataset directory:", DATA_DIR.resolve())

## 2. Dataset description and discovery

The project uses the GTZAN music genre collection. The code below supports either of these common layouts:

- `Data/genres_original/<genre>/*.wav`
- `Data/<genre>/*.wav`

The notebook searches for the ten expected genre folders and creates a table of discovered audio files. The dataset itself is not included in this notebook and must be placed locally.

In [ ]:
def discover_audio_files(dataset_dir):
    """Find supported audio files in the expected GTZAN genre folders."""
    root = Path(dataset_dir)
    records = []
    extensions = {".wav", ".mp3", ".au", ".flac", ".ogg"}

    for genre in GENRES:
        folder = root / genre
        if not folder.exists():
            folder = root / "genres_original" / genre
        if not folder.exists():
            continue
        for path in sorted(folder.iterdir()):
            if path.is_file() and path.suffix.lower() in extensions:
                records.append({"path": path, "genre": genre})
    return records

records = discover_audio_files(DATA_DIR)
if not records:
    raise FileNotFoundError(
        f"No audio files found under {DATA_DIR.resolve()}. "
        "Expected genre folders such as Data/genres_original/blues/*.wav. "
        "Update DATA_DIR to your dataset location."
    )

dataset_df = pd.DataFrame(records)
print(f"Total audio files discovered: {len(dataset_df)}")
display(dataset_df.head())

## 3. Exploratory data analysis

First inspect the number of tracks in each genre. A class-count chart helps identify missing or unevenly represented classes before training.

In [ ]:
genre_counts = dataset_df["genre"].value_counts().reindex(GENRES, fill_value=0)
display(genre_counts.rename("track_count").to_frame())

plt.figure(figsize=(11, 5))
sns.barplot(x=genre_counts.index, y=genre_counts.values)
plt.title("Number of Audio Tracks per Genre")
plt.xlabel("Genre")
plt.ylabel("Number of tracks")
plt.xticks(rotation=35)
plt.tight_layout()
plt.show()

### 3.1 Inspect an example audio track

Listen to one example and inspect its waveform. This provides a basic view of the audio signal before feature extraction.

In [ ]:
sample_row = dataset_df.iloc[0]
sample_path = sample_row["path"]
y_sample, sr_sample = librosa.load(
    str(sample_path), sr=SAMPLE_RATE, mono=True, duration=DURATION_SECONDS
)
print("Example file:", sample_path)
print("Genre:", sample_row["genre"])
print("Sample rate:", sr_sample)
print("Duration (seconds):", round(len(y_sample) / sr_sample, 2))
display(Audio(y_sample, rate=sr_sample))

plt.figure(figsize=(13, 3))
librosa.display.waveshow(y_sample, sr=sr_sample)
plt.title(f"Waveform – {sample_row['genre'].title()}")
plt.xlabel("Time (seconds)")
plt.tight_layout()
plt.show()

## 4. Audio preprocessing

Each track is loaded as mono audio at **22,050 Hz** and processed as a maximum of **30 seconds**. Short tracks are zero-padded and longer tracks are truncated to the configured duration. This produces a consistent input length for feature extraction.

The shared `extract_mfcc()` function in `src/audio_features.py` performs this preprocessing and returns a fixed-size feature tensor.

In [ ]:
print("Sample rate:", SAMPLE_RATE)
print("Target duration:", DURATION_SECONDS, "seconds")
print("MFCC coefficients:", N_MFCC)
print("Expected MFCC frames:", MAX_FRAMES)

sample_features = extract_mfcc(sample_path)
print("Feature shape:", sample_features.shape)
print("Feature dtype:", sample_features.dtype)
print("All feature values finite:", np.isfinite(sample_features).all())

## 5. MFCC feature extraction and visualization

Mel-Frequency Cepstral Coefficients (MFCCs) represent the short-term spectral envelope of an audio signal. They provide a compact representation of timbral information that can be used by a classifier.

This project extracts **40 MFCC coefficients** using Librosa. The extracted matrix is normalized coefficient-wise across time, then given a channel dimension for the 2D CNN.

In [ ]:
plt.figure(figsize=(13, 4))
librosa.display.specshow(
    sample_features[:, :, 0],
    x_axis="time",
    sr=SAMPLE_RATE,
    hop_length=512,
    cmap="magma",
)
plt.colorbar(format="%+2.0f")
plt.title(f"Normalized MFCC Features – {sample_row['genre'].title()}")
plt.xlabel("Frame")
plt.ylabel("MFCC coefficient")
plt.tight_layout()
plt.show()

## 6. Build the feature and label arrays

Each readable track is converted into an MFCC tensor. Genre names are mapped to integer class IDs using the fixed `GENRES` order. Files that cannot be decoded are reported and skipped; the final class counts should be checked after extraction.

In [ ]:
X_list, y_list, processed_paths, skipped_files = [], [], [], []

for index, record in enumerate(records, start=1):
    path = record["path"]
    genre = record["genre"]
    try:
        feature = extract_mfcc(path)
        X_list.append(feature)
        y_list.append(GENRES.index(genre))
        processed_paths.append(str(path))
    except Exception as exc:
        skipped_files.append((str(path), str(exc)))
        print(f"[SKIP] {path}: {exc}")

    if index % 50 == 0 or index == len(records):
        print(f"Processed {index}/{len(records)} files")

X = np.asarray(X_list, dtype=np.float32)
y = np.asarray(y_list, dtype=np.int64)

if len(X) == 0:
    raise RuntimeError("No audio files could be processed. Check the dataset and audio codecs.")
if len(np.unique(y)) < 2:
    raise RuntimeError("At least two different genres are required for classification.")

print("Feature array shape:", X.shape)
print("Label array shape:", y.shape)
print("Successfully processed:", len(X))
print("Skipped:", len(skipped_files))
print("Non-finite feature values:", int((~np.isfinite(X)).sum()))

processed_counts = pd.Series([GENRES[i] for i in y]).value_counts().reindex(GENRES, fill_value=0)
display(processed_counts.rename("processed_tracks").to_frame())

## 7. Encode labels and split the dataset

The target labels are integer encoded according to the `GENRES` list. The data is divided into **80% training** and **20% validation** sets using stratification, so each genre is represented proportionally in both subsets.

This is a random track-level split. It should be interpreted as a project benchmark, not an artist-independent evaluation. Dataset duplicates or related recordings can affect generalization estimates.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    stratify=y,
)

print(f"Training samples: {len(X_train)} ({len(X_train) / len(X):.1%})")
print(f"Validation samples: {len(X_val)} ({len(X_val) / len(X):.1%})")
print("Input shape:", X_train.shape[1:])
print("Number of classes:", len(GENRES))

## 8. Build the Convolutional Neural Network

The CNN learns local patterns from the MFCC feature maps. The architecture contains three convolutional blocks (32, 64 and 128 filters), batch normalization, max pooling, global average pooling, a 128-unit dense layer, dropout, and a 10-class softmax output.

In [ ]:
def build_model(input_shape, num_classes):
    inputs = tf.keras.Input(shape=input_shape)

    x = tf.keras.layers.Conv2D(32, (3, 3), padding="same", activation="relu")(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    x = tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    x = tf.keras.layers.Conv2D(128, (3, 3), padding="same", activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs, name="GENREation_CNN")
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_model(
    input_shape=(N_MFCC, MAX_FRAMES, 1),
    num_classes=len(GENRES),
)
model.summary()

## 9. Train the CNN

The model is trained using Adam and sparse categorical cross-entropy. Early stopping restores the weights from the best validation-loss epoch, the learning-rate scheduler reduces the learning rate when validation loss stops improving, and model checkpointing saves the best validation-accuracy model.

You can adjust `EPOCHS` and `BATCH_SIZE` according to available compute resources. The values below match the project's training script defaults.

In [ ]:
EPOCHS = 30
BATCH_SIZE = 8

best_model_path = ARTIFACT_DIR / "music_genre_cnn.keras"

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(best_model_path),
        monitor="val_accuracy",
        save_best_only=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    ),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

## 10. Visualize training history

Training and validation accuracy/loss curves help identify whether the model is learning and whether there may be overfitting or underfitting.

In [ ]:
history_df = pd.DataFrame(history.history)
display(history_df.tail())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history.history["accuracy"], label="Training accuracy")
axes[0].plot(history.history["val_accuracy"], label="Validation accuracy")
axes[0].set_title("Training and Validation Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

axes[1].plot(history.history["loss"], label="Training loss")
axes[1].plot(history.history["val_loss"], label="Validation loss")
axes[1].set_title("Training and Validation Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

history_df.to_csv(ARTIFACT_DIR / "training_history.csv", index=False)

## 11. Evaluate the best saved model

Evaluation is performed on the held-out validation set. The notebook reports accuracy, precision, recall and F1-score for each genre, followed by a confusion matrix. These are the actual results from the current execution; do not substitute previously reported results if a new training run produces different values.

In [ ]:
if best_model_path.exists():
    evaluated_model = tf.keras.models.load_model(best_model_path)
else:
    evaluated_model = model

val_probabilities = evaluated_model.predict(X_val, batch_size=BATCH_SIZE, verbose=0)
y_pred = np.argmax(val_probabilities, axis=1)

val_accuracy = accuracy_score(y_val, y_pred)
report_text = classification_report(
    y_val,
    y_pred,
    labels=list(range(len(GENRES))),
    target_names=GENRES,
    zero_division=0,
)
report_dict = classification_report(
    y_val,
    y_pred,
    labels=list(range(len(GENRES))),
    target_names=GENRES,
    output_dict=True,
    zero_division=0,
)

print(f"Validation accuracy: {val_accuracy:.2%}")
print(report_text)

In [ ]:
cm = confusion_matrix(y_val, y_pred, labels=list(range(len(GENRES))))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=GENRES,
    yticklabels=GENRES,
    ax=ax,
)
ax.set_title("GENREation – Confusion Matrix")
ax.set_xlabel("Predicted genre")
ax.set_ylabel("Actual genre")
plt.xticks(rotation=35)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

cm_path = ARTIFACT_DIR / "confusion_matrix.png"
fig.savefig(cm_path, dpi=160, bbox_inches="tight")
plt.close(fig)

## 12. Save model, labels and evaluation results

The trained model and supporting metadata are saved in the `artifacts` directory. The genre order in `labels.json` must remain consistent with the output neuron order of the model; the Streamlit app uses this file to interpret predictions.

In [ ]:
labels_path = ARTIFACT_DIR / "labels.json"
evaluation_path = ARTIFACT_DIR / "evaluation.json"

with open(labels_path, "w", encoding="utf-8") as f:
    json.dump({"genres": GENRES}, f, indent=2)

with open(evaluation_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "validation_accuracy": float(val_accuracy),
            "classification_report": report_dict,
            "validation_samples": int(len(y_val)),
            "training_samples": int(len(y_train)),
            "genres": GENRES,
        },
        f,
        indent=2,
    )

print("Saved files:")
for path in [best_model_path, labels_path, evaluation_path, cm_path, ARTIFACT_DIR / "training_history.csv"]:
    print("-", path.resolve())

## 13. Test on an unseen audio file

Set `TEST_AUDIO_PATH` to a separate audio file that was not used in the training/validation data if possible. The same `extract_mfcc()` function is used here so the inference features match the model's training input format.

If no separate test file is available, the cell will use one validation track as a demonstration. That fallback is not an independent unseen-track test and should be described accordingly in a report.

In [ ]:
# Replace with the path to an external audio file for a true unseen-audio test.
TEST_AUDIO_PATH = None  # Example: Path("sample_music/my_track.mp3")

if TEST_AUDIO_PATH is not None:
    test_audio_path = Path(TEST_AUDIO_PATH)
    is_independent_test = True
else:
    test_audio_path = Path(processed_paths[0])
    is_independent_test = False

if not test_audio_path.exists():
    raise FileNotFoundError(f"Audio file not found: {test_audio_path}")

test_features = extract_mfcc(test_audio_path)
test_input = np.expand_dims(test_features, axis=0)
test_probabilities = evaluated_model.predict(test_input, verbose=0)[0]
ranked = np.argsort(test_probabilities)[::-1]

predicted_id = int(ranked[0])
predicted_genre = GENRES[predicted_id]
confidence = float(test_probabilities[predicted_id])

print("Audio file:", test_audio_path)
print("Evaluation type:", "Independent audio file" if is_independent_test else "Validation-set demonstration")
print("Predicted genre:", predicted_genre.title())
print(f"Model confidence: {confidence:.2%}")

print("\nTop 3 predictions:")
for rank, class_id in enumerate(ranked[:3], start=1):
    print(f"{rank}. {GENRES[int(class_id)].title()}: {test_probabilities[class_id]:.2%}")

In [ ]:
y_test_audio, sr_test_audio = librosa.load(
    str(test_audio_path),
    sr=SAMPLE_RATE,
    mono=True,
    duration=DURATION_SECONDS,
)
print("Audio duration:", round(len(y_test_audio) / sr_test_audio, 2), "seconds")
display(Audio(y_test_audio, rate=sr_test_audio))

mel = librosa.feature.melspectrogram(
    y=y_test_audio,
    sr=sr_test_audio,
    n_fft=2048,
    hop_length=512,
    n_mels=128,
)
mel_db = librosa.power_to_db(mel, ref=np.max)

plt.figure(figsize=(12, 4))
librosa.display.specshow(
    mel_db,
    sr=sr_test_audio,
    hop_length=512,
    x_axis="time",
    y_axis="mel",
    cmap="magma",
)
plt.colorbar(format="%+2.0f dB")
plt.title("Log-Mel Spectrogram")
plt.tight_layout()
plt.show()

## 14. Results interpretation

Use the metrics and plots produced above to discuss the model's behaviour. In particular:

- **Accuracy** is the proportion of validation tracks classified correctly.
- **Precision** indicates how often predictions for a genre are correct.
- **Recall** indicates how many actual tracks of a genre are identified.
- **F1-score** balances precision and recall.
- **Confusion matrix** shows which genres are most often confused with one another.

Do not interpret a high softmax score on one uploaded track as proof that the model is accurate. Softmax confidence is a model output, whereas validation accuracy and class-wise metrics measure performance across a labelled dataset.

## 15. Limitations and future improvements

### Limitations
- GTZAN is a relatively small benchmark dataset and has known data-quality and duplication concerns.
- The random track-level split is not necessarily artist-independent.
- Music can contain multiple genres, while this model predicts one class.
- Audio quality, background noise, short clips and music outside the training distribution can affect predictions.
- Softmax confidence is not necessarily a calibrated probability of correctness.

### Future improvements
- Audit duplicate or overlapping tracks and use artist-independent evaluation where reliable metadata is available.
- Compare MFCC features with log-Mel spectrogram inputs.
- Experiment with segment-based training and inference while preventing track-level data leakage.
- Apply carefully controlled augmentation and hyperparameter tuning.
- Compare CNN results with classical models such as SVM and Random Forest.
- Evaluate confidence calibration and performance on a separate external dataset.

## 16. Conclusion

GENREation demonstrates an end-to-end music genre classification workflow using audio preprocessing, MFCC feature extraction and a CNN implemented with TensorFlow/Keras. The trained model is evaluated using a stratified validation split and can be integrated with the Streamlit interface for interactive audio prediction. The notebook, model checkpoint, label mapping and evaluation artifacts together document the implementation and its measured performance.